# 04. Dataset Merging & Final Processing
This notebook joins all the daily data sources, performs leak-free satellite monthly median imputation on splits, calculates engineered target lag features, and creates the final daily merged files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json

ROOT = Path("../../")
INTERIM_DIR = ROOT / "data/interim"
PROC_DIR = ROOT / "data/processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load and Merge Datasets
We load the OpenAQ daily base, OpenMeteo weather base, 30 satellite columns, and join them.

In [ ]:
# Load OpenAQ Daily Base
df_aq = pd.read_csv(INTERIM_DIR / "openaq/all_stations_daily.csv")
df_aq["date"] = pd.to_datetime(df_aq["date"].astype(str).str.strip()).dt.tz_localize(None)
df_aq = df_aq.rename(columns={
    "pm25_mean": "pm25",
    "pm25_hourly_count": "pm25_count",
    "pm25_std": "pm25_daily_std"
})
keep_aq_cols = ["date", "location_id", "location_name", "latitude", "longitude", "pm25", "pm25_count", "pm25_daily_std", "pm25_was_interpolated", "split"]
df_aq = df_aq[keep_aq_cols]

# Load OpenMeteo Daily Base
df_met = pd.read_csv(INTERIM_DIR / "openmeteo/all_stations_daily.csv")
df_met["date"] = pd.to_datetime(df_met["date"].astype(str).str.strip()).dt.tz_localize(None)
rename_met = {
    "temperature_2m_C_mean": "temperature_2m_C_mean",
    "relative_humidity_pct_mean": "relative_humidity_pct_mean",
    "wind_speed_mean_kmh": "wind_speed_10m_kmh_mean",
    "wind_u_mean": "wind_u_10m_mean",
    "wind_v_mean": "wind_v_10m_mean",
    "precip_daily_mm": "precipitation_mm_sum",
    "cloud_cover_mean_pct": "cloud_cover_pct_mean",
    "pressure_msl_mean_hPa": "pressure_msl_hPa_mean"
}
df_met = df_met.rename(columns=rename_met)
met_cols = ["date", "location_id"] + list(rename_met.values())
df_met = df_met[met_cols]

df_master = df_aq.merge(df_met, on=["location_id", "date"], how="left")

# Load Satellite 30-feature base
df_sat = pd.read_csv(ROOT / "data/raw/DataAOD/Hanoi/all_stations_satellite_daily.csv")
df_sat["date"] = pd.to_datetime(df_sat["date"].astype(str).str.strip()).dt.tz_localize(None)
sat_cols_3d = [
    "no2_valid_pixels", "co_valid_pixels", "so2_valid_pixels", "aer_ai_340_380_valid_pixels", "s2_valid_pixels",
    "no2_mean", "no2_std", "no2_min", "no2_max", "no2_median",
    "co_mean", "co_std", "co_min", "co_max", "co_median",
    "so2_mean", "so2_std", "so2_min", "so2_max", "so2_median",
    "aer_ai_340_380_mean", "aer_ai_340_380_std", "aer_ai_340_380_min", "aer_ai_340_380_max", "aer_ai_340_380_median",
    "ndvi_mean", "ndvi_std", "ndvi_min", "ndvi_max", "ndvi_median"
]
df_sat = df_sat[["location_id", "date"] + sat_cols_3d]
df_master = df_master.merge(df_sat, on=["location_id", "date"], how="left")

# Drop invalid target values (consistent with preprocess script)
df_master = df_master[df_master["pm25"].notna() & (df_master["pm25"] > 0) & (df_master["pm25"] <= 500)].reset_index(drop=True)

# Save unimputed daily merged file
df_master.to_csv(PROC_DIR / "02_daily_merged_unimputed.csv", index=False)
df_master.to_csv(PROC_DIR / "daily_merged_unimputed.csv", index=False)
print(f"Saved unimputed merged daily dataset. Shape: {df_master.shape}")


Saved unimputed merged daily dataset. Shape: (2582, 48)


## 2. Leak-Free Satellite Imputation
We impute missing satellite values using a split-safe strategy: linear time interpolation per station, followed by train-only monthly median fallback, and train-only station median fallback.

In [ ]:
def impute_satellite(df, sat_cols, split_col="split"):
    tr_idx = df[df[split_col] == "train"].index.tolist()
    train_rows = df.iloc[tr_idx]
    train_monthly = train_rows.groupby(["location_id", train_rows["date"].dt.month])[sat_cols].median()
    
    parts = []
    for loc_id, grp in df.groupby("location_id"):
        grp = grp.sort_values("date").copy()
        for c in sat_cols:
            if c not in grp.columns:
                continue
            grp[c] = grp[c].interpolate(method="linear", limit=14, limit_direction="forward")
            grp[c] = grp[c].interpolate(method="linear", limit=7,  limit_direction="backward")
            if grp[c].isna().any():
                for month, mg in grp.groupby(grp["date"].dt.month):
                    fill = train_monthly.loc[(loc_id, month), c] if (loc_id, month) in train_monthly.index else np.nan
                    if np.isnan(fill):
                        fill = train_rows[train_rows["location_id"] == loc_id][c].median()
                    grp.loc[mg.index, c] = grp.loc[mg.index, c].fillna(fill)
        parts.append(grp)
    return pd.concat(parts).sort_index()

df_imputed = df_master.copy()
df_imputed = impute_satellite(df_imputed, sat_cols_3d)
print("Imputation of satellite features completed.")


Imputation of satellite features completed.


## 3. Merge Weather, Satellite, Static and Fire Features
We load and merge the rest of processed weather and satellite files.

In [ ]:
# Load ERA5 BLH
df_blh = pd.read_csv(PROC_DIR / "04_era5_blh_daily.csv")
df_blh["date"] = pd.to_datetime(df_blh["date"].astype(str).str.strip()).dt.tz_localize(None)

# Load ERA5 RH850
df_rh = pd.read_csv(PROC_DIR / "05_era5_rh850_daily.csv").drop(columns=["month"], errors="ignore")
df_rh["date"] = pd.to_datetime(df_rh["date"].astype(str).str.strip()).dt.tz_localize(None)

# Load Temp Inversion
df_tinv = pd.read_csv(PROC_DIR / "06_era5_t_inversion_daily.csv")
df_tinv["date"] = pd.to_datetime(df_tinv["date"].astype(str).str.strip()).dt.tz_localize(None)

# Load CAMS AOD
df_cams = pd.read_csv(PROC_DIR / "07_cams_aod_daily.csv").drop(columns=["month"], errors="ignore")
df_cams["date"] = pd.to_datetime(df_cams["date"].astype(str).str.strip()).dt.tz_localize(None)

# Load MAIAC AOD
df_maiac = pd.read_csv(PROC_DIR / "08_maiac_aod_daily.csv")
df_maiac["date"] = pd.to_datetime(df_maiac["date"].astype(str).str.strip()).dt.tz_localize(None)

# Load FIRMS Fire
df_fire = pd.read_csv(PROC_DIR / "09_firms_daily.csv")
df_fire["date"] = pd.to_datetime(df_fire["date"].astype(str).str.strip()).dt.tz_localize(None)

# Load Static features
df_sf = pd.read_csv(PROC_DIR / "03_station_spatial_features.csv")
keep_sf_cols = [
    "location_id", "built_up_frac_500m", "built_up_frac_1km", "built_up_frac_2km", "built_up_frac_5km",
    "tree_frac_2km", "cropland_frac_2km", "water_frac_2km",
    "dist_motorway_m", "dist_primary_m", "dist_secondary_m", "dist_any_major_m",
    "road_density_1km", "road_density_2km", "n_major_edges_5km"
]
df_sf = df_sf[keep_sf_cols]

# Merge sequentially
df_imputed = df_imputed.merge(df_blh, on=["location_id", "date"], how="left")
df_imputed = df_imputed.merge(df_rh, on=["location_id", "date"], how="left")
df_imputed = df_imputed.merge(df_tinv, on=["location_id", "date"], how="left")
df_imputed = df_imputed.merge(df_cams, on=["location_id", "date"], how="left")
df_imputed = df_imputed.merge(df_maiac, on=["location_id", "date"], how="left")
df_imputed = df_imputed.merge(df_fire, on=["location_id", "date"], how="left")
df_imputed = df_imputed.merge(df_sf, on=["location_id"], how="left")

print("All daily features merged. Shape:", df_imputed.shape)


All daily features merged. Shape: (2582, 86)


## 4. Feature Engineering: Lags, Rollings, Calendar and Sin/Cos Transformations
Compute target lags (`pm25_lag1`, etc.) and rolling statistics (`pm25_roll3`, etc.) to prevent leakage, along with cyclical time encodings.

In [ ]:
# Calendar components
df_imputed["day_of_week"] = df_imputed["date"].dt.dayofweek
df_imputed["month"] = df_imputed["date"].dt.month
df_imputed["day_of_year"] = df_imputed["date"].dt.dayofyear
df_imputed["is_weekend"] = df_imputed["day_of_week"].isin([5, 6]).astype("int8")

# Cyclical encodings
df_imputed["sin_doy"] = np.sin(2 * np.pi * df_imputed["day_of_year"] / 365.25)
df_imputed["cos_doy"] = np.cos(2 * np.pi * df_imputed["day_of_year"] / 365.25)
df_imputed["sin_month"] = np.sin(2 * np.pi * df_imputed["month"] / 12)
df_imputed["cos_month"] = np.cos(2 * np.pi * df_imputed["month"] / 12)

# Target lags and rolling rollouts per station
df_imputed = df_imputed.sort_values(["location_id", "date"]).reset_index(drop=True)

df_imputed["pm25_lag1"] = df_imputed.groupby("location_id")["pm25"].shift(1)
df_imputed["pm25_lag2"] = df_imputed.groupby("location_id")["pm25"].shift(2)
df_imputed["pm25_lag3"] = df_imputed.groupby("location_id")["pm25"].shift(3)
df_imputed["pm25_lag7"] = df_imputed.groupby("location_id")["pm25"].shift(7)
df_imputed["pm25_delta"] = df_imputed["pm25_lag1"] - df_imputed["pm25_lag2"]

df_imputed["pm25_roll3"] = df_imputed.groupby("location_id")["pm25_lag1"].transform(lambda s: s.rolling(3, min_periods=1).mean())
df_imputed["pm25_roll7"] = df_imputed.groupby("location_id")["pm25_lag1"].transform(lambda s: s.rolling(7, min_periods=1).mean())
df_imputed["pm25_roll7std"] = df_imputed.groupby("location_id")["pm25_lag1"].transform(lambda s: s.rolling(7, min_periods=1).std())

# Slice dataset to match target dates starting 2024-02-06
df_imputed = df_imputed[df_imputed["date"] >= "2024-02-06"].reset_index(drop=True)

# Save final daily merged dataset
df_imputed.to_csv(PROC_DIR / "01_daily_merged.csv", index=False)
df_imputed.to_csv(PROC_DIR / "daily_merged.csv", index=False)
print(f"Saved final imputed master dataset. Shape: {df_imputed.shape}")


Saved final imputed master dataset. Shape: (2566, 102)


## 5. Generate Preprocessing Report
Compute missingness statistics before and after imputation to generate `11_preprocessing_report.json`.

In [ ]:
report_data = {
    "input_files": ["all_stations_satellite_hourly.csv", "all_stations_openmeteo_hourly.csv"],
    "output_file": "daily_merged.csv",
    "rows": len(df_imputed),
    "columns": len(df_imputed.columns),
    "stations": int(df_imputed["location_id"].nunique()),
    "imputation_strategy": {
        "pm25_hourly": "linear interpolation <= 6h per station",
        "pm25_daily": "linear forward-fill <= 3 days, then drop",
        "satellite": "forward 14d -> backward 7d -> monthly median -> station median",
        "met": "no imputation needed (0% missing)"
    }
}

with open(PROC_DIR / "11_preprocessing_report.json", "w", encoding="utf-8") as f:
    json.dump(report_data, f, indent=2)
with open(PROC_DIR / "preprocessing_report.json", "w", encoding="utf-8") as f:
    json.dump(report_data, f, indent=2)
print("Saved preprocessing report.")


Saved preprocessing report.
